# JS06 Assignment - Support Vector Machine
    
Setiap bagian soal dipisahkan ke dalam cell masing-masing.

## Soal 1 - Persiapan data voice.csv

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

DATA_DIR = Path.cwd()
if not (DATA_DIR / 'voice.csv').exists() and (DATA_DIR / 'JS06' / 'voice.csv').exists():
    DATA_DIR = DATA_DIR / 'JS06'

voice = pd.read_csv(DATA_DIR / 'voice.csv')
X = voice.drop(columns='label')
y = voice['label']

print(f'Jumlah data: {len(voice)}')
print(f'Jumlah fitur: {X.shape[1]}')
print('Distribusi label:')
print(y.value_counts())

## Soal 1a - Split data 70:30 dan 80:20

In [ ]:
splits = {
    '70:30': train_test_split(
        X, y, test_size=0.30, random_state=42, stratify=y
    ),
    '80:20': train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )
}

for split_name, (X_train, X_test, y_train, y_test) in splits.items():
    print(
        f'{split_name}: train={len(X_train)}, test={len(X_test)}, '
        f'kelas train={y_train.nunique()}, kelas test={y_test.nunique()}'
    )

## Soal 1a(i-iii) - Model linear, polynomial, dan RBF

In [ ]:
models = {
    'linear': SVC(kernel='linear'),
    'polynomial': SVC(kernel='poly', degree=3),
    'RBF': SVC(kernel='rbf')
}

results = []

for split_name, (X_train, X_test, y_train, y_test) in splits.items():
    for kernel_name, svc in models.items():
        model = Pipeline([
            ('scaler', StandardScaler()),
            ('svc', svc)
        ])
        model.fit(X_train, y_train)
        prediction = model.predict(X_test)
        results.append({
            'split': split_name,
            'kernel': kernel_name,
            'accuracy': accuracy_score(y_test, prediction)
        })

q1_results = pd.DataFrame(results)
q1_results

## Soal 1b - Tabel performa berdasarkan accuracy

In [ ]:
performance_table = q1_results.pivot(
    index='kernel', columns='split', values='accuracy'
).reindex(index=['linear', 'polynomial', 'RBF'], columns=['70:30', '80:20'])
performance_table.style.format('{:.4f}')

## Soal 2 - Persiapan data citra day/night

Struktur folder yang digunakan:

```text
images/
├── training/
│   ├── day/
│   └── night/
└── test/
    ├── day/
    └── night/
```

Jika folder citra berada di lokasi lain, ubah nilai `IMAGE_DIR`.

In [ ]:
import cv2
from sklearn.model_selection import GridSearchCV

IMAGE_DIR = Path.cwd() / 'images'
if not IMAGE_DIR.exists() and (Path.cwd() / 'JS06' / 'images').exists():
    IMAGE_DIR = Path.cwd() / 'JS06' / 'images'

def load_images(image_dir):
    images, labels = [], []
    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}

    for label_name in ('day', 'night'):
        label_dir = Path(image_dir) / label_name
        if not label_dir.exists():
            raise FileNotFoundError(f'Folder tidak ditemukan: {label_dir}')
        for image_path in sorted(label_dir.iterdir()):
            if image_path.suffix.lower() not in valid_extensions:
                continue
            image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
            if image is None:
                raise ValueError(f'Citra tidak dapat dibaca: {image_path}')
            images.append(image)
            labels.append(label_name)
    return images, np.array(labels)
images, image_labels = load_images(IMAGE_DIR / 'training')
print(f'Jumlah citra: {len(images)}')
print(pd.Series(image_labels).value_counts())

## Soal 2 - Ekstraksi histogram features

In [ ]:
def histogram_feature(image, bins=32):
    histogram = cv2.calcHist([image], [0], None, [bins], [0, 256]).ravel()
    histogram = histogram / (histogram.sum() + 1e-12)
    return histogram
X_image = np.array([histogram_feature(image) for image in images])
y_image = np.where(image_labels == 'day', 1, 0)
X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(
    X_image, y_image, test_size=0.20, random_state=42, stratify=y_image
)
print(f'Bentuk fitur histogram: {X_image.shape}')
print(f'Train: {X_train_img.shape[0]}, test: {X_test_img.shape[0]}')

## Soal 2 - SVM RBF dan hyperparameter tuning

In [ ]:
rbf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='rbf'))
])
    
parameter_grid = {
    'svc__C': [0.1, 1, 10, 100],
    'svc__gamma': ['scale', 0.001, 0.01, 0.1]
}
search = GridSearchCV(
    rbf_pipeline,
    parameter_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
search.fit(X_train_img, y_train_img)
y_pred_img = search.predict(X_test_img)
accuracy_img = accuracy_score(y_test_img, y_pred_img)
print('Parameter terbaik:', search.best_params_)
print(f'Best CV accuracy: {search.best_score_:.4f}')
print(f'Accuracy test 80:20: {accuracy_img:.4f}')

## Soal 2 - Kesimpulan

In [ ]:
print(
    f'Model SVM dengan kernel RBF memperoleh accuracy test '
    f'{accuracy_img:.4f} menggunakan split 80:20.'
)
print('Label 1 = day, label 0 = night.')